# Kaggle training — Akkadian → English (ByT5)

Вариант для Kaggle Notebooks (квота 30 GPU-ч/нед, сессии до 12 ч). Рассчитан на
**Save & Run All (Commit)** — весь ноутбук выполняется батчем.
Чекпоинты пушатся на HF Hub, резюм после обрыва — автоматический.

**Важно:** веса пишутся в эфемерную `/kaggle/temp` (НЕ в `/kaggle/working`), иначе
Kaggle при коммите часами сохраняет ~7 ГБ чекпоинтов как output. В output остаётся
только маленький `submission.csv`.

**Настройка (один раз):**
1. Справа **Accelerator → GPU P100** (или T4 x2) и **Internet → On**.
2. **Input → + Add Input** → Competitions → **Deep Past Initiative: Machine Translation**.
3. **Add-ons → Secrets** → `HF_TOKEN` (Write) и `WANDB_API_KEY`, Attach.

**Режимы** (`TRAIN`): `True` — обучить по `CONFIG` и оценить; `False` — только оценка модели с Hub.

In [ ]:
CONFIG = "configs/exp1_norm.yaml"  # <- какой эксперимент
TRAIN = True                       # <- False = только оценка модели с Hub
BRANCH = "ml-dev"

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # на T4 x2 учимся на одной GPU
OUT_DIR = "/kaggle/temp/byt5"            # эфемерно: не попадёт в output коммита
!nvidia-smi -L

In [ ]:
!rm -rf /kaggle/working/repo
!git clone --branch {BRANCH} https://github.com/ObjoradDdd/ml-hits-3-lab.git /kaggle/working/repo
%cd /kaggle/working/repo/ml
!pip install -q -e . sacrebleu

In [ ]:
# секреты -> окружение; логин в HF (нужно для приватного репозитория с весами)
import yaml
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
for key in ("HF_TOKEN", "WANDB_API_KEY"):
    try:
        os.environ[key] = secrets.get_secret(key)
    except Exception:
        print("no secret:", key)

from huggingface_hub import login, whoami
login(os.environ["HF_TOKEN"])
HF_USER = whoami()["name"]
RUN_NAME = yaml.safe_load(open(CONFIG))["run_name"]
HUB_ID = f"{HF_USER}/akkadian-{RUN_NAME}"
FINAL = OUT_DIR + "/final"
# что грузим на оценке: локальную обученную модель или модель с Hub
MODEL = FINAL if TRAIN else HUB_ID
print("HUB_ID:", HUB_ID, "| оценка из:", MODEL)

In [ ]:
# данные соревнования уже подключены как Input — копируем нужные csv
COMP = "/kaggle/input/competitions/deep-past-initiative-machine-translation"
!mkdir -p data && cp {COMP}/train.csv {COMP}/test.csv \
    {COMP}/published_texts.csv {COMP}/Sentences_Oare_FirstWord_LinNum.csv data/
!python -m akkadian_nmt.data_prep --data_dir=./data --out_dir=./data/processed

In [ ]:
# обучение (только если TRAIN=True). Веса -> /kaggle/temp (не в output).
# Резюм с Hub после обрыва — автоматический.
if TRAIN:
    !python -m akkadian_nmt.train --config={CONFIG} \
        --output_dir={OUT_DIR} --push_to_hub=True --hub_model_id={HUB_ID}
else:
    print("TRAIN=False — обучение пропущено, оценка модели с Hub")

In [ ]:
# оценка на испорченном dev: greedy vs beam {1,4,8} (эксперимент 2)
!python -m akkadian_nmt.evaluate beam_sweep --model_dirs={MODEL} --max_samples=200

In [ ]:
# полный метрический набор (BLEU + chrF++ + geo-mean + COMET) — beam=4
# ВАЖНО: запускаем ПОСЛЕДНИМ — unbabel-comet тянет transformers<5.0 и портит окружение.
# Можно закомментировать, если не нужен COMET (экономит ~15-20 мин).
!pip install -q unbabel-comet
!python -m akkadian_nmt.evaluate run --model_dirs={MODEL} --num_beams=4 --comet=True \
    --out_file=/kaggle/working/dev_predictions.json